<a href="https://colab.research.google.com/github/Kaveesha-Vihanga/DS_Project/blob/component-3/Model_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
pip install xmltodict pandas scikit-learn transformers torch tqdm

In [4]:
import xmltodict
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer
from tqdm import tqdm
import torch
import re
import os

# ------------------------------------------------------------
# 1. Load and parse the XML file robustly
# ------------------------------------------------------------
xml_file = "/content/drive/MyDrive/DSGP Component 3/Hotels_review_Testing.xml"

if not os.path.exists(xml_file):
    raise FileNotFoundError(f"XML file not found: {xml_file}")

with open(xml_file, 'r', encoding='utf-8') as f:
    xml_string = f.read()

def parse_xml(xml_string):
    data = xmltodict.parse(xml_string)
    reviews_list = data['Reviews']['Review']
    if isinstance(reviews_list, dict):
        reviews_list = [reviews_list]

    total_reviews = len(reviews_list)
    reviews_with_opinions = 0
    records = []

    for review in reviews_list:
        text = review['text']
        opinions_elem = review.get('Opinions')

        # If Opinions element is missing or None (empty tag), skip this review
        if not opinions_elem:
            continue

        # Get the Opinion(s)
        opinion_list = opinions_elem.get('Opinion')
        if opinion_list is None:
            continue   # Opinions element exists but has no Opinion children

        # Normalize to list
        if isinstance(opinion_list, dict):
            opinion_list = [opinion_list]

        # This review has at least one opinion
        reviews_with_opinions += 1

        for op in opinion_list:
            records.append({
                'text': text,
                'category': op['@category'],
                'polarity': op['@polarity'].lower()
            })

    return records, total_reviews, reviews_with_opinions

records, total_reviews, reviews_with_opinions = parse_xml(xml_string)
df = pd.DataFrame(records)

# ------------------------------------------------------------
# 2. Dataset summary (your requested format)
# ------------------------------------------------------------
print("\n" + "="*60)
print("DATASET SUMMARY")
print("="*60)
print(f"  o Dataset used: {os.path.basename(xml_file)}")
print(f"  o Total reviews in file: {total_reviews}")
print(f"  o Reviews containing at least one opinion: {reviews_with_opinions}")
print(f"  o Total (text, category, polarity) samples: {len(df)}")
print(f"  o Sentiment distribution among annotations:")
pos = (df['polarity'] == 'positive').sum()
neg = (df['polarity'] == 'negative').sum()
neu = (df['polarity'] == 'neutral').sum()
print(f"     - Positive: {pos}")
print(f"     - Negative: {neg}")
print(f"     - Neutral:  {neu}")
print("\n  o Unique aspect categories found:")
for cat in sorted(df['category'].unique()):
    print(f"     - {cat}")
print("="*60)

# ------------------------------------------------------------
# 3. Load models and define prediction functions
# ------------------------------------------------------------
device = 0 if torch.cuda.is_available() else -1
print(f"\nUsing device: {'cuda' if device==0 else 'cpu'}")

# ---- Model 1: BART MNLI ----
print("\nLoading facebook/bart-large-mnli...")
bart_pipe = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=device)

def predict_bart(text):
    candidate_labels = ['positive', 'negative', 'neutral']
    result = bart_pipe(text, candidate_labels)
    return result['labels'][0]

# ---- Model 2: SieBERT ----
print("Loading siebert/sentiment-roberta-large-english...")
siebert_pipe = pipeline("sentiment-analysis", model="siebert/sentiment-roberta-large-english", device=device)

def predict_siebert(text):
    result = siebert_pipe(text)[0]
    label = result['label'].lower()
    if 'positive' in label:
        return 'positive'
    elif 'negative' in label:
        return 'negative'
    else:
        return 'neutral'

# ---- Model 3: AOS Triplet ----
print("Loading tafseer-nayeem/aspect-opinion-sentiment_AOS-triplet...")
aos_tokenizer = AutoTokenizer.from_pretrained("tafseer-nayeem/aspect-opinion-sentiment_AOS-triplet")
aos_model = AutoModelForSeq2SeqLM.from_pretrained("tafseer-nayeem/aspect-opinion-sentiment_AOS-triplet")
aos_model.to(device if device >= 0 else 'cpu')
aos_model.eval()

def extract_triplets(text):
    input_text = f"extract triplets: {text}"
    inputs = aos_tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512)
    if device >= 0:
        inputs = {k: v.to('cuda') for k, v in inputs.items()}
    with torch.no_grad():
        outputs = aos_model.generate(**inputs, max_new_tokens=128)
    decoded = aos_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded

def parse_triplets(output_string):
    triplets = []
    parts = output_string.split(';')
    for part in parts:
        match = re.search(r'aspect:\s*(.*?),\s*opinion:\s*(.*?),\s*sentiment:\s*(.*?)(;|$)', part.strip(), re.IGNORECASE)
        if match:
            aspect = match.group(1).strip()
            opinion = match.group(2).strip()
            sentiment = match.group(3).strip().lower()
            # Clean punctuation
            aspect = re.sub(r'[^\w\s]', '', aspect)
            opinion = re.sub(r'[^\w\s]', '', opinion)
            sentiment = re.sub(r'[^\w\s]', '', sentiment)
            if sentiment in ['positive', 'negative', 'neutral']:
                triplets.append((aspect, opinion, sentiment))
    return triplets

# Build a simple mapping from category to keyword (the part before '#')
category_to_keyword = {cat: cat.split('#')[0].lower() for cat in df['category'].unique()}

def predict_aos(text, gold_category):
    output = extract_triplets(text)
    triplets = parse_triplets(output)
    keyword = category_to_keyword.get(gold_category, gold_category.split('#')[0].lower())

    for aspect, opinion, sentiment in triplets:
        if keyword in aspect.lower():
            return sentiment
    return 'neutral'

# ------------------------------------------------------------
# 4. Run evaluation for each model
# ------------------------------------------------------------
models = [
    ("facebook/bart-large-mnli", predict_bart),
    ("siebert/sentiment-roberta-large-english", predict_siebert),
    ("tafseer-nayeem/aspect-opinion-sentiment_AOS-triplet", predict_aos)
]

for model_name, predict_func in models:
    print(f"\n{'='*60}")
    print(f"Evaluating {model_name}")
    print('='*60)

    y_true = []
    y_pred = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=model_name[:30]):
        text = row['text']
        category = row['category']
        true_sent = row['polarity']

        if model_name == "tafseer-nayeem/aspect-opinion-sentiment_AOS-triplet":
            pred_sent = predict_func(text, category)
        else:
            pred_sent = predict_func(text)

        y_true.append(true_sent)
        y_pred.append(pred_sent)

    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, digits=3))
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.3f}")


DATASET SUMMARY
  o Dataset used: Hotels_review_Testing.xml
  o Total reviews in file: 1847
  o Reviews containing at least one opinion: 1496
  o Total (text, category, polarity) samples: 7454
  o Sentiment distribution among annotations:
     - Positive: 6079
     - Negative: 1146
     - Neutral:  149

  o Unique aspect categories found:
     - FACILITIES#CLEANLINESS
     - FACILITIES#COMFORT
     - FACILITIES#DESIGN_FEATURES
     - FACILITIES#GENERAL
     - FACILITIES#MISCELLANEOUS
     - FACILITIES#PRICES
     - FACILITIES#QUALITY
     - FOOD_DRINKS#MISCELLANEOUS
     - FOOD_DRINKS#PRICES
     - FOOD_DRINKS#QUALITY
     - FOOD_DRINKS#STYLE_OPTIONS
     - HOTEL#CLEANLINESS
     - HOTEL#COMFORT
     - HOTEL#DESIGN_FEATURES
     - HOTEL#GENERAL
     - HOTEL#MISCELLANEOUS
     - HOTEL#PRICES
     - HOTEL#QUALITY
     - LOCATION#GENERAL
     - POLARITY#NEGATIVE
     - POLARITY#POSITIVE
     - ROOMS#CLEANLINESS
     - ROOMS#COMFORT
     - ROOMS#DESIGN_FEATURES
     - ROOMS#GENERAL
     -

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

...

siebert/sentiment-roberta-larg: 100%|██████████| 7454/7454 [03:45<00:00, 33.04it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metr


Classification Report:
              precision    recall  f1-score   support

    conflict      0.000     0.000     0.000        80
    negative      0.707     0.548     0.618      1146
     neutral      0.000     0.000     0.000       149
    positive      0.895     0.966     0.929      6079

    accuracy                          0.872      7454
   macro avg      0.400     0.379     0.387      7454
weighted avg      0.838     0.872     0.853      7454

Accuracy: 0.872

Evaluating tafseer-nayeem/aspect-opinion-sentiment_AOS-triplet


tafseer-nayeem/aspect-opinion-: 100%|██████████| 7454/7454 [1:53:10<00:00,  1.10it/s]


Classification Report:
              precision    recall  f1-score   support

    conflict      0.000     0.000     0.000        80
    negative      0.000     0.000     0.000      1146
     neutral      0.020     1.000     0.039       149
    positive      0.000     0.000     0.000      6079

    accuracy                          0.020      7454
   macro avg      0.005     0.250     0.010      7454
weighted avg      0.000     0.020     0.001      7454

Accuracy: 0.020



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
